In [ ]:
# ===== Celda 0: Subir y descomprimir el subconjunto etiquetado =====
import os, zipfile
from google.colab import files

# --- TODO 1: subí tu archivo .zip cuando aparezca el botón "Elegir archivos" ---
# Se abre un selector. Elegí el zip que armaste con tus imágenes etiquetadas.
# Ejemplo de lo que vas a subir: "subset_etiquetado.zip"
subidos = files.upload()

# --- TODO 2: poné el nombre EXACTO de tu zip, igual al que subiste ---
# Ejemplo: si subiste "subset_etiquetado.zip", dejá esa línea tal cual.
#          si tu archivo se llama "morfologia_imgs.zip", cambialo por ese.
NOMBRE_ZIP = "training-dataset.zip"   # <-- cambialo si tu zip se llama distinto

# Carpeta donde van a quedar las imágenes descomprimidas (no hace falta tocar)
DESTINO_IMAGENES = "/content/images"

# Descompresión
os.makedirs(DESTINO_IMAGENES, exist_ok=True)
with zipfile.ZipFile(NOMBRE_ZIP, 'r') as z:
    z.extractall(DESTINO_IMAGENES)

# Verificación: cuántas imágenes quedaron y algunos nombres de ejemplo
extensiones = (".png", ".jpg", ".jpeg")
imagenes = []
for raiz, _, archivos in os.walk(DESTINO_IMAGENES):
    for a in archivos:
        if a.lower().endswith(extensiones):
            imagenes.append(os.path.join(raiz, a))

print(f"Imágenes encontradas tras descomprimir: {len(imagenes)}")
print("Ejemplos:", [os.path.basename(p) for p in imagenes[:5]])

# --- TODO 3 (posible ajuste): revisá la RUTA real de las imágenes ---
# A veces el zip trae una subcarpeta interna. Si arriba ves rutas como
#   /content/images/subset_etiquetado/PAT_123.png
# en vez de
#   /content/images/PAT_123.png
# entonces en la Celda 1 tenés que poner:
#   IMAGES_DIR = "/content/images/subset_etiquetado"
# Si las imágenes están sueltas directamente en /content/images, no cambies nada.

In [ ]:
# ===== Celda 1: Configuración e imports =====
# Notebook de entrenamiento - SEADD capa de visión (solo morfología)
# Probado con TensorFlow 2.15+ (el de Colab por defecto sirve).

import os, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

# --- TODO 4: subí también tu labels.csv (botón de archivos del panel izquierdo)
#             o con files.upload(), y poné su ruta acá ---
LABELS_CSV = "/content/labels.csv"   # <-- ajustá si lo dejaste en otra ruta

# --- TODO 5: que coincida con el TODO 3 de arriba ---
IMAGES_DIR = "/content/images"       # o "/content/images/subset_etiquetado"

# Taxonomía EXACTA de la base de conocimiento (el orden define los índices de clase)
MORPH_CLASSES = ["macula", "papula", "ampolla", "escama", "engrosamiento"]

IMG_SIZE = 224
BATCH = 16                 # chico, porque el dataset etiquetado es chico
BACKBONE = "efficientnet"  # "efficientnet" (B0) o "mobilenet" (V3Small)
EPOCHS_FROZEN = 15
EPOCHS_FINETUNE = 10
N_CLASSES = len(MORPH_CLASSES)

In [ ]:
# ===== Celda 2: Cargar y validar el CSV de etiquetas =====
df = pd.read_csv(LABELS_CSV)
df.columns = [c.strip().lower() for c in df.columns]
assert {"image_filename", "morphology"}.issubset(df.columns), \
    "El CSV debe tener columnas image_filename y morphology"

# Limpieza de valores (espacios, mayúsculas inconsistentes)
df["morphology"] = df["morphology"].astype(str).str.strip()

# Filas con etiqueta fuera de la taxonomía o vacías -> se reportan y se descartan
valid = df["morphology"].isin(MORPH_CLASSES)
if (~valid).any():
    print("Filas con morfología no reconocida (se descartan):")
    print(df.loc[~valid, "morphology"].value_counts())
df = df[valid].copy()

# Verificar que las imágenes existan en disco
df["path"] = df["image_filename"].apply(lambda f: os.path.join(IMAGES_DIR, f))
exists = df["path"].apply(os.path.isfile)
if (~exists).any():
    print(f"{(~exists).sum()} imágenes del CSV no se encontraron en disco (se descartan).")
df = df[exists].copy()

# Índice de clase
df["label"] = df["morphology"].map({c: i for i, c in enumerate(MORPH_CLASSES)})

print(f"\nTotal de imágenes etiquetadas usables: {len(df)}")
print("Conteo por clase:")
print(df["morphology"].value_counts().reindex(MORPH_CLASSES).fillna(0).astype(int))

In [ ]:
# ===== Celda 3: Split estratificado train/val/test =====
# Estratificado para que cada clase aparezca en los tres conjuntos.
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}: {len(d)} imágenes")
    print(d["morphology"].value_counts().reindex(MORPH_CLASSES).fillna(0).astype(int).to_dict())

In [ ]:
# ===== Celda 4: Pipelines tf.data + aumentos + pesos de clase =====
if BACKBONE == "efficientnet":
    preprocess = tf.keras.applications.efficientnet.preprocess_input
else:
    preprocess = tf.keras.applications.mobilenet_v3.preprocess_input

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label

# Aumentos: como el color ya NO es objetivo, podemos variar brillo/contraste libremente.
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
], name="aumentos")

def make_ds(d, training=False):
    ds = tf.data.Dataset.from_tensor_slices((d["path"].values, d["label"].values))
    if training:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

# Pesos de clase para el desbalance (clases raras como Ampolla pesan más)
cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES),
                          y=train_df["label"].values)
class_weight = {i: w for i, w in enumerate(cw)}
print("Pesos de clase:", {MORPH_CLASSES[i]: round(w, 2) for i, w in class_weight.items()})

In [ ]:
# ===== Celda 5: Modelo (transfer learning, una sola cabeza) =====
if BACKBONE == "efficientnet":
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
else:
    base = tf.keras.applications.MobileNetV3Small(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False  # congelado en la primera fase

inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(N_CLASSES, activation="softmax", name="morfologia")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# ===== Celda 6: Entrenar la cabeza (backbone congelado) =====
hist1 = model.fit(train_ds, validation_data=val_ds,
                  epochs=EPOCHS_FROZEN, class_weight=class_weight)

In [ ]:
# ===== Celda 7: Fine-tuning (descongelar parte del backbone) =====
base.trainable = True
for layer in base.layers[:-30]:   # descongelamos solo las últimas capas
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # LR bajo en fine-tuning
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = model.fit(train_ds, validation_data=val_ds,
                  epochs=EPOCHS_FINETUNE, class_weight=class_weight)

In [ ]:
# ===== Celda 8: Evaluación POR CLASE (no una exactitud global) =====
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(axis=1)

print("Reporte por clase (precision / recall / F1):\n")
print(classification_report(y_true, y_pred, target_names=MORPH_CLASSES,
                            labels=np.arange(N_CLASSES), zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(N_CLASSES))
ConfusionMatrixDisplay(cm, display_labels=MORPH_CLASSES).plot(xticks_rotation=45)
plt.title("Matriz de confusión - Morfología"); plt.tight_layout(); plt.show()

In [ ]:
# ===== Celda 9: Exportar para inferencia en CPU + función de predicción =====
model.save("seadd_morfologia.keras")          # formato Keras
model.export("seadd_morfologia_savedmodel")    # SavedModel (servible en CPU)

# Opcional: exportar a ONNX (descomentá si lo necesitás)
# !pip -q install tf2onnx onnx
# !python -m tf2onnx.convert --saved-model seadd_morfologia_savedmodel --output seadd_morfologia.onnx --opset 13

def predecir_morfologia(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = preprocess(tf.cast(img, tf.float32))[None, ...]
    p = model.predict(img, verbose=0)[0]
    i = int(p.argmax())
    return {"morfologia": MORPH_CLASSES[i], "confianza": float(p[i])}

# Ejemplo:
# print(predecir_morfologia("/content/images/ejemplo.png"))